# NB_Silver_Dim_Geolocation — Bronze to Silver Transformation (Upgraded v2 — Educational & Production-Ready)

## Section 1 — Purpose, Business Context & Engineering Decisions
- **Source Table:** `Cloud_Retail_Data_Lakehouse_v2.dbo.dim_geolocation` → ADLS Bronze Parquet exposed via Fabric Shortcut (`Files/bronze/reference/dim_geolocation/`).
- **Bronze Grain:** Multiple coordinate observations per Brazilian 5-digit ZIP prefix (`1,000,163` raw rows across `19,015` distinct ZIP prefixes).
- **Target Silver Table:** `silver.geolocation_zip_reference` (**strictly 1:1 — one canonical row per `geolocation_zip_code_prefix`**, `19,015` rows).
- **Quarantine Table:** `silver._quarantine_dim_geolocation` (stores out-of-Brazil coordinate outliers and invalid ZIP/state rows for auditability).
- **Execution Environment:** Built for **Microsoft Fabric (`synapse_pyspark` kernel)** where `spark` is pre-configured automatically. Do **not** add `%pip install pyspark`.

### Key Data Quality Findings Driving This Notebook (`CONFIRMED` on `1,000,163` source rows)
1. **Massive Exact Full-Row Duplication (`261,831` rows / `26.18%`):** Leaving exact duplicates in place before aggregation distorts both `percentile_approx` median coordinates and modal city/state vote counts. We deduplicate valid coordinate observations first (`720,469` unique observations after text standardization).
2. **`29` Out-of-Brazil Coordinate Outliers (`4` ZIPs Lose ALL Rows if Not Backfilled):** `29` rows fall outside Brazil's bounding box (`lat` in `[-35, 6]`, `lng` in `[-75, -30]`), with coordinates landing in Europe, North America, and the Philippines (`95130` at `14.59, 121.11` — `18,195.6 km` error!). Dropping them removes **`4` valid Brazilian ZIPs** (`18243`, `78131`, `83252`, `95130`), including `1` customer in `83252`. We quarantine the `29` corrupted rows AND backfill those `4` ZIPs with externally verified coordinates so all **`19,015`** ZIPs remain intact.
3. **Portuguese Accent Splitting (`8,006` ZIPs) & Encoding/Symbol Corruptions:** Raw cities have `8,011` spellings (`5,967` after accent folding), HTML/URL-encoded apostrophes (`%26apos%3b`), garbled UTF-8 (`sa£o paulo`), leading symbols (`...arraial do cabo`, `* cidade`), and bare abbreviations (`sp`, `rj`, `bh`, `saopaulo`).
4. **`8` Multi-State ZIPs & `545` Multi-City ZIPs:** Deterministic `Window` mode selection resolves all `8` multi-state ZIPs (`02116`, `04011`, `21550`, `23056`, `72915`, `78557`, `79750`, `80630`) to Brazil's true state (`100%` verified against Correios), protecting **`74` customers** and **`1` seller**.

> **WARNING:** **DO NOT EXECUTE THE WRITE STEP UNTIL THE TRANSFORMATION HAS BEEN REVIEWED AND APPROVED.** (`WRITE_ENABLED = False` by default.)

## Section 2 — Configuration, Parameters & Verified Coordinate Backfill
### What does this cell do?
- **`WRITE_ENABLED = False`:** Safety lock that allows a full **Dry Run** without modifying any Delta tables in Fabric.
- **Relative Fabric Shortcut Path (`Files/bronze/...`):** Avoids hardcoded workspace/lakehouse GUIDs; relies on `Retail_Analytics_Lakehouse` attached as Default Lakehouse.
- **`SILVER_SCHEMA = "silver"`:** Explicitly defined so `CREATE SCHEMA IF NOT EXISTS silver` runs safely before `.saveAsTable(...)`.
- **`VERIFIED_OUT_OF_BRAZIL_BACKFILL`:** Audited coordinates (verified in `Geolocation_Anomaly_Audit.xlsx`) for the `4` ZIP prefixes whose raw Olist rows were 100% outside Brazil, ensuring zero loss of ZIP coverage (`19,011 + 4 = 19,015`).

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# =============================================================================
# 1. SAFETY LOCK — Run once as Dry Run (False), then set True to persist Delta
# =============================================================================
WRITE_ENABLED = False

# =============================================================================
# 2. PIPELINE PARAMETERS (Relative Fabric Shortcut Path)
# =============================================================================
BRONZE_DIR = "Files/bronze/reference/dim_geolocation"
SOURCE_FILE_NAME = "dim_geolocation_20261121_013331_b0e93f82-84a0-4668-9a8c-9ae1da7dd183.parquet"
BRONZE_PATH = f"{BRONZE_DIR}/{SOURCE_FILE_NAME}"

INGESTION_RUN_ID = "b0e93f82-84a0-4668-9a8c-9ae1da7dd183"
SOURCE_SYSTEM = "sqlserver_olist"
SOURCE_TABLE = "dbo.dim_geolocation"

SILVER_SCHEMA = "silver"
SILVER_TARGET_TABLE = f"{SILVER_SCHEMA}.geolocation_zip_reference"
QUARANTINE_TARGET_TABLE = f"{SILVER_SCHEMA}._quarantine_dim_geolocation"

# Official 27 Brazilian Federal Units (UF)
VALID_BR_STATES = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
]

# Verified external coordinates for the 4 ZIP prefixes whose raw Olist rows were 100% out of Brazil
# Format: (zip_prefix, canonical_city, state, verified_lat, verified_lng, raw_quarantined_count)
VERIFIED_OUT_OF_BRAZIL_BACKFILL = [
    ("18243", "bom retiro da esperanca", "SP", -23.54759, -48.30551, 1),
    ("78131", "varzea grande", "MT", -15.64667, -56.13250, 1),
    ("83252", "ilha dos valadares", "PR", -25.53500, -48.49194, 2),
    ("95130", "santa lucia do piai", "RS", -29.24165, -51.02058, 1),
]

print(f"Configured BRONZE_PATH       : {BRONZE_PATH}")
print(f"Configured SILVER_TARGET     : {SILVER_TARGET_TABLE}")
print(f"Configured QUARANTINE_TARGET : {QUARANTINE_TARGET_TABLE}")
print(f"WRITE_ENABLED                : {WRITE_ENABLED}")

## Section 3 — Read Bronze Parquet & Capture Source File Lineage
### What does this cell do?
- Reads the exact Bronze Parquet snapshot for `dim_geolocation` (`1,000,163` rows).
- Attaches `_source_file` via `F.input_file_name()` for end-to-end lineage tracking.
- Caches `df_bronze` in Spark memory because we perform multiple passes (schema check, quarantine split, deduplication, and reconciliation).

In [ ]:
df_bronze = (
    spark.read
    .format("parquet")
    .load(BRONZE_PATH)
    .withColumn("_source_file", F.input_file_name())
)

df_bronze.cache()
bronze_count = df_bronze.count()

print(f"Bronze Raw Row Count: {bronze_count:,}")
df_bronze.printSchema()

## Section 4 — Stage 1: Schema Validation (Fail-Fast Contract Check)
### What does this cell do?
- Verifies that all 5 required columns (`geolocation_zip_code_prefix`, `geolocation_lat`, `geolocation_lng`, `geolocation_city`, `geolocation_state`) exist in the Parquet file before running any transformation.
- Raises a `ValueError` immediately on schema drift.

In [ ]:
REQUIRED_COLUMNS = [
    "geolocation_zip_code_prefix",
    "geolocation_lat",
    "geolocation_lng",
    "geolocation_city",
    "geolocation_state",
]

missing_columns = [c for c in REQUIRED_COLUMNS if c not in df_bronze.columns]
if missing_columns:
    raise ValueError(f"Schema validation failed — missing required columns: {missing_columns}")

print("Stage 1 Schema Validation: PASSED")

## Section 5 — Stage 2: Text Standardization, Encoding Repair & Portuguese Accent Folding
### Why is every step here necessary?
1. **Preserve `geolocation_city_raw`:** Keeps the trimmed, lowercase, single-spaced original city string for auditability.
2. **Repair Corrupted Encodings & Symbols:**
   - Decodes URL/HTML entity `%26apos%3b` back to `'` (fixes `são joão do pau d%26apos%3balho` and `lambari d%26apos%3boeste`).
   - Fixes garbled UTF-8/Latin-1 `sa£o paulo` → `sao paulo`.
   - Strips leading noise prefixes (`...` in `...arraial do cabo` and `* ` in `* cidade`).
3. **Portuguese Accent Folding (`F.translate`):** Collapses `áàâãäéèêëíìîïóòôõöúùûüç` to ASCII equivalents (`aaaaaeeeeiiiiooooouuuuc`) natively in Spark without slow Python UDFs — resolving **`8,006` ZIP prefixes** whose vote counts were split by accent differences (`são paulo` vs `sao paulo`).
4. **Normalize Known Bare Abbreviations:** Maps `sp` → `sao paulo`, `rj` → `rio de janeiro`, `bh` → `belo horizonte`, and `saopaulo` → `sao paulo`, while keeping real numbered Brazilian cities like `4º centenario` and `riacho fundo 2` intact.

In [ ]:
ACCENTED_CHARS = "áàâãäéèêëíìîïóòôõöúùûüç"
ASCII_CHARS    = "aaaaaeeeeiiiiooooouuuuc"

city_clean_expr = F.lower(F.trim(F.regexp_replace(F.col("geolocation_city").cast("string"), r"\s+", " ")))
city_clean_expr = F.regexp_replace(city_clean_expr, r"%26apos%3b", "'")
city_clean_expr = F.regexp_replace(city_clean_expr, r"sa£o paulo", "sao paulo")
city_clean_expr = F.regexp_replace(city_clean_expr, r"^\.{2,}|^\*\s*", "")
city_clean_expr = F.translate(city_clean_expr, ACCENTED_CHARS, ASCII_CHARS)
city_clean_expr = F.trim(F.regexp_replace(city_clean_expr, r"\s+", " "))

df_std = (
    df_bronze
    .withColumn("geolocation_zip_code_prefix", F.trim(F.col("geolocation_zip_code_prefix").cast("string")))
    .withColumn("geolocation_lat", F.col("geolocation_lat").cast("double"))
    .withColumn("geolocation_lng", F.col("geolocation_lng").cast("double"))
    .withColumn("geolocation_city_raw", F.lower(F.trim(F.regexp_replace(F.col("geolocation_city").cast("string"), r"\s+", " "))))
    .withColumn("geolocation_city_norm", city_clean_expr)
    .withColumn(
        "geolocation_city",
        F.when(F.col("geolocation_city_norm") == "sp", F.lit("sao paulo"))
         .when(F.col("geolocation_city_norm") == "saopaulo", F.lit("sao paulo"))
         .when(F.col("geolocation_city_norm") == "rj", F.lit("rio de janeiro"))
         .when(F.col("geolocation_city_norm") == "bh", F.lit("belo horizonte"))
         .otherwise(F.col("geolocation_city_norm"))
    )
    .withColumn("geolocation_state", F.upper(F.trim(F.col("geolocation_state").cast("string"))))
    .drop("geolocation_city_norm")
)

print("Stage 2 Standardization complete. Distinct cleaned cities:", df_std.select("geolocation_city").distinct().count())

## Section 6 — Stage 3: Row-Level Validation & Out-of-Brazil Quarantine Split
### What does this cell do?
- Validates every raw observation against 4 strict rules:
  1. `geolocation_zip_code_prefix` must be exactly 5 digits (`^\d{5}$`).
  2. `geolocation_state` must belong to Brazil's 27 official UF codes (`VALID_BR_STATES`).
  3. `geolocation_lat` must be non-null and within Brazil's latitude bounding box `[-35.0, 6.0]`.
  4. `geolocation_lng` must be non-null and within Brazil's longitude bounding box `[-75.0, -30.0]`.
- Diverts failing rows (**`29` expected** — coordinates in Europe, North America, Argentina, and the Philippines) into `df_quarantine` with a semicolon-separated `dq_reject_reason` and full audit columns.

In [ ]:
zip_invalid = F.col("geolocation_zip_code_prefix").isNull() | (~F.col("geolocation_zip_code_prefix").rlike(r"^\d{5}$"))
state_invalid = F.col("geolocation_state").isNull() | (~F.col("geolocation_state").isin(*VALID_BR_STATES))
lat_out_bbox = F.col("geolocation_lat").isNull() | (~F.col("geolocation_lat").between(-35.0, 6.0))
lng_out_bbox = F.col("geolocation_lng").isNull() | (~F.col("geolocation_lng").between(-75.0, -30.0))

df_classified = df_std.withColumn(
    "dq_reject_reason",
    F.concat_ws(
        ";",
        F.when(zip_invalid, F.lit("INVALID_ZIP_PREFIX")),
        F.when(state_invalid, F.lit("INVALID_BR_STATE")),
        F.when(lat_out_bbox, F.lit("LAT_OUTSIDE_BRAZIL_BBOX")),
        F.when(lng_out_bbox, F.lit("LNG_OUTSIDE_BRAZIL_BBOX")),
    )
)

df_quarantine = (
    df_classified
    .filter(F.col("dq_reject_reason") != "")
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("source_table", F.lit(SOURCE_TABLE))
    .withColumn("ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumnRenamed("_source_file", "source_file")
    .withColumn("quarantined_at", F.current_timestamp())
)

df_valid_raw = df_classified.filter(F.col("dq_reject_reason") == "").drop("dq_reject_reason")

valid_raw_count = df_valid_raw.count()
quarantine_count = df_quarantine.count()

print(f"Valid Raw Observations : {valid_raw_count:,} (Expected: 1,000,134)")
print(f"Quarantined Outliers   : {quarantine_count:,} (Expected: 29)")

## Section 7 — Stage 4: Exact Full-Row Deduplication (Removing the `261,831+` Duplicates)
### Why deduplicate BEFORE computing centroids and modal city/state?
- `dim_geolocation` contains **`261,831` exact raw duplicate rows (`26.18%`)** — and **`720,469` unique observations** after text normalization and outlier removal.
- If we run `percentile_approx` or `groupBy().count()` on un-deduplicated rows, repeated identical coordinate rows unfairly bias the median centroid and city/state vote counts.
- We also preserve `raw_row_count` per ZIP alongside `dedup_coordinate_count` for full transparency.

In [ ]:
# Count total valid raw observations per ZIP before deduplication
df_raw_zip_counts = (
    df_valid_raw
    .groupBy("geolocation_zip_code_prefix")
    .agg(F.count("*").alias("source_raw_row_count"))
)

# Remove exact duplicate coordinate/city/state observations
df_valid_dedup = df_valid_raw.dropDuplicates([
    "geolocation_zip_code_prefix",
    "geolocation_lat",
    "geolocation_lng",
    "geolocation_city",
    "geolocation_state",
])

df_valid_dedup.cache()
dedup_count = df_valid_dedup.count()
print(f"Deduplicated Valid Observations: {dedup_count:,} (Removed {valid_raw_count - dedup_count:,} duplicate observations)")

## Section 8 — Stage 5: Aggregate to Canonical 1:1 ZIP Reference (`19,011` Valid-Coord ZIPs)
### How does this prevent Cartesian fan-out and resolve multi-state/multi-city ZIPs?
1. **Modal City & State (`Window`):** Counts deduplicated observations per `(zip, city, state)` and picks the highest-frequency pair deterministically (`orderBy(desc("obs_count"), asc("geolocation_city"))`), resolving all **`8` multi-state ZIPs** (`02116`, `04011`, `21550`, `23056`, `72915`, `78557`, `79750`, `80630`) to Brazil's true state.
2. **Quality Flags:** Adds `geo_multi_state_flag` (flags the `8` multi-state ZIPs) and `geo_multi_city_flag` (flags the `~545` multi-city ZIPs).
3. **Median Centroid (`percentile_approx(..., 0.5)`):** Resistant to remaining intra-Brazil GPS jitter compared to `avg()`.

In [ ]:
# 1. Modal city & state per ZIP
df_city_state_votes = (
    df_valid_dedup
    .groupBy("geolocation_zip_code_prefix", "geolocation_city", "geolocation_state")
    .agg(F.count("*").alias("obs_count"))
)

w_mode = Window.partitionBy("geolocation_zip_code_prefix").orderBy(F.desc("obs_count"), F.asc("geolocation_city"))
df_modal = (
    df_city_state_votes
    .withColumn("rn", F.row_number().over(w_mode))
    .filter(F.col("rn") == 1)
    .drop("rn", "obs_count")
)

# 2. Multi-state & multi-city diagnostic flags per ZIP
df_zip_ambiguity = (
    df_valid_dedup
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        (F.countDistinct("geolocation_state") > 1).alias("geo_multi_state_flag"),
        (F.countDistinct("geolocation_city") > 1).alias("geo_multi_city_flag"),
    )
)

# 3. Median centroid coordinates & deduplicated observation count per ZIP
df_coords = (
    df_valid_dedup
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        F.expr("percentile_approx(geolocation_lat, 0.5)").alias("centroid_lat"),
        F.expr("percentile_approx(geolocation_lng, 0.5)").alias("centroid_lng"),
        F.count("*").alias("dedup_coordinate_count"),
    )
)

df_geo_olist = (
    df_coords
    .join(df_modal, on="geolocation_zip_code_prefix", how="inner")
    .join(df_zip_ambiguity, on="geolocation_zip_code_prefix", how="inner")
    .join(df_raw_zip_counts, on="geolocation_zip_code_prefix", how="inner")
    .withColumn("coordinate_source", F.lit("OLIST_MEDIAN_CENTROID"))
    .withColumn("has_valid_coordinates", F.lit(True))
)

print("Olist Valid-Coord ZIP Reference Count:", df_geo_olist.count(), "(Expected: 19,011)")

## Section 9 — Stage 6: Backfill the `4` Out-of-Brazil-Only ZIPs (`19,011 + 4 = 19,015` Total ZIPs)
### Why backfill these `4` ZIP prefixes (`18243`, `78131`, `83252`, `95130`)?
- All raw observations for these `4` ZIPs had corrupted coordinates outside Brazil (`6,723 km` to `18,195 km` error) and were quarantined in Stage 3.
- Without this backfill, downstream `LEFT JOIN`s (including `1` real customer in `83252` — Ilha dos Valadares, PR) would lose coordinates completely.
- We append the `4` externally verified rows (only if not already present in `df_geo_olist`) tagged with `coordinate_source = 'EXTERNAL_VERIFIED_BACKFILL'` for complete auditability.

In [ ]:
backfill_rows = [
    (z, float(lat), float(lng), 0, city, st, False, False, int(raw_cnt), "EXTERNAL_VERIFIED_BACKFILL", True)
    for (z, city, st, lat, lng, raw_cnt) in VERIFIED_OUT_OF_BRAZIL_BACKFILL
]

df_backfill = spark.createDataFrame(
    backfill_rows,
    schema=df_geo_olist.select(
        "geolocation_zip_code_prefix",
        "centroid_lat",
        "centroid_lng",
        "dedup_coordinate_count",
        "geolocation_city",
        "geolocation_state",
        "geo_multi_state_flag",
        "geo_multi_city_flag",
        "source_raw_row_count",
        "coordinate_source",
        "has_valid_coordinates",
    ).schema,
).join(df_geo_olist.select("geolocation_zip_code_prefix"), on="geolocation_zip_code_prefix", how="left_anti")

df_geo_combined = df_geo_olist.select(
    "geolocation_zip_code_prefix",
    "geolocation_city",
    "geolocation_state",
    "centroid_lat",
    "centroid_lng",
    "source_raw_row_count",
    "dedup_coordinate_count",
    "geo_multi_state_flag",
    "geo_multi_city_flag",
    "coordinate_source",
    "has_valid_coordinates",
).unionByName(
    df_backfill.select(
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "geolocation_state",
        "centroid_lat",
        "centroid_lng",
        "source_raw_row_count",
        "dedup_coordinate_count",
        "geo_multi_state_flag",
        "geo_multi_city_flag",
        "coordinate_source",
        "has_valid_coordinates",
    )
)

# Mandatory 1:1 Uniqueness Check before building final Silver table
dup_zips = df_geo_combined.groupBy("geolocation_zip_code_prefix").count().filter(F.col("count") > 1).count()
if dup_zips > 0:
    raise ValueError(f"1:1 ZIP uniqueness violated: {dup_zips} duplicate ZIPs found!")

print("Combined 1:1 ZIP Reference Count:", df_geo_combined.count(), "(Expected: 19,015)")

## Section 10 — Stage 7: Build Final Silver DataFrame & Pre-Write Reconciliation
### What does this cell do?
- Attaches full audit & lineage columns (`source_system`, `source_table`, `ingestion_run_id`, `silver_processed_at`).
- Verifies strict reconciliation:
  - `bronze_count == valid_raw_count + quarantine_count` (`1,000,163 == 1,000,134 + 29`)
  - `silver_count == 19,015` (`19,011` Olist median centroids + `4` verified backfills)
  - All `19,015` centroids fall inside Brazil's bounding box (`lat` in `[-35, 6]`, `lng` in `[-75, -30]`).

In [ ]:
df_silver = (
    df_geo_combined
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("source_table", F.lit(SOURCE_TABLE))
    .withColumn("ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumn("silver_processed_at", F.current_timestamp())
)

silver_count = df_silver.count()
multi_state_cnt = df_silver.filter(F.col("geo_multi_state_flag")).count()
multi_city_cnt = df_silver.filter(F.col("geo_multi_city_flag")).count()
backfill_cnt = df_silver.filter(F.col("coordinate_source") == "EXTERNAL_VERIFIED_BACKFILL").count()
out_after_fix = df_silver.filter(
    (~F.col("centroid_lat").between(-35.0, 6.0)) | (~F.col("centroid_lng").between(-75.0, -30.0))
).count()

print(f"Bronze Raw Count           : {bronze_count:,}")
print(f"Valid Raw Observations     : {valid_raw_count:,}")
print(f"Quarantined Outliers       : {quarantine_count:,} (Expected: 29)")
print(f"Silver 1:1 ZIPs Count      : {silver_count:,} (Expected: 19,015)")
print(f"  -> Olist Median Centroids: {silver_count - backfill_cnt:,} (Expected: 19,011)")
print(f"  -> Verified Backfills    : {backfill_cnt:,} (Expected: 4)")
print(f"  -> Multi-State ZIPs      : {multi_state_cnt:,} (Expected: 8)")
print(f"  -> Multi-City ZIPs       : {multi_city_cnt:,} (Expected: 545)")
print(f"  -> Out-of-Brazil Centroid: {out_after_fix:,} (Expected: 0)")

if bronze_count != valid_raw_count + quarantine_count:
    raise ValueError(f"Raw reconciliation failed: {bronze_count} != {valid_raw_count} + {quarantine_count}")
if out_after_fix != 0:
    raise ValueError(f"Found {out_after_fix} centroids still outside Brazil bounding box!")

df_silver.printSchema()
print("Stage 7 Reconciliation: PASSED")

## Section 11 — Proposed Silver & Quarantine Write (DISABLED BY DEFAULT)
> **DO NOT EXECUTE THE WRITE STEP UNTIL THE TRANSFORMATION HAS BEEN REVIEWED AND APPROVED.**

### What does this cell do?
- When `WRITE_ENABLED = True`:
  - Runs `CREATE SCHEMA IF NOT EXISTS silver` to prevent `SCHEMA_NOT_FOUND` in schemas-enabled Fabric Lakehouses.
  - Writes `silver.geolocation_zip_reference` (`19,015` 1:1 rows) as a managed Delta Catalog table.
  - Appends the `29` quarantined out-of-Brazil rows to `silver._quarantine_dim_geolocation`.

In [ ]:
if WRITE_ENABLED:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

    (
        df_silver.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(SILVER_TARGET_TABLE)
    )
    print(f"Successfully wrote {silver_count:,} 1:1 ZIP rows to {SILVER_TARGET_TABLE}")

    if quarantine_count > 0:
        (
            df_quarantine.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(QUARANTINE_TARGET_TABLE)
        )
        print(f"Appended {quarantine_count:,} outlier rows to {QUARANTINE_TARGET_TABLE}")
else:
    print("WRITE_ENABLED is False — dry validation only; no Silver or Quarantine data was written.")

## Section 12 — Post-Write Catalog Validation & Cache Cleanup
### What does this cell do?
- After `WRITE_ENABLED = True`, reads back `spark.table("silver.geolocation_zip_reference")` from the Catalog to confirm:
  - Exact row count (`19,015`).
  - Zero duplicate `geolocation_zip_code_prefix` keys (`1:1` guarantee).
  - Displays the `4` externally backfilled ZIP rows for visual verification.
- Unpersists cached DataFrames (`df_bronze` and `df_valid_dedup`) to release executor memory.

In [ ]:
if WRITE_ENABLED:
    df_check = spark.table(SILVER_TARGET_TABLE)
    final_count = df_check.count()
    if final_count != silver_count:
        raise ValueError(f"Post-write count mismatch: expected {silver_count}, actual {final_count}")

    final_dups = df_check.groupBy("geolocation_zip_code_prefix").count().filter(F.col("count") > 1).count()
    if final_dups > 0:
        raise ValueError(f"Post-write 1:1 violation: {final_dups} duplicate ZIP prefixes found!")

    print(f"POST-WRITE CATALOG VALIDATION PASSED: {final_count:,} 1:1 ZIP rows, 0 duplicates.")
    (
        df_check
        .filter(F.col("coordinate_source") == "EXTERNAL_VERIFIED_BACKFILL")
        .select("geolocation_zip_code_prefix", "geolocation_city", "geolocation_state", "centroid_lat", "centroid_lng", "coordinate_source")
        .show(truncate=False)
    )
else:
    print("Post-write validation skipped because WRITE_ENABLED is False.")

df_bronze.unpersist()
df_valid_dedup.unpersist()